# 20260317 convert Feronia samples Seurat objects to Scanpy objects
This runs in seu4_label_transfer env

In [1]:
# Check the conda env running, it's important that you're running in the correct conda env
# Don't change anything here
Sys.getenv("CONDA_DEFAULT_ENV")

[1] "seu4_label_transfer"

In [2]:
# Check Seurat version, it's important that you're using the correct version of Seurat
# scKB_copilot: Seurat 3; seu4_label_transfer: Seurat 4; muscat: Seurat 5
# Don't change anything here
packageVersion("Seurat")

[1] ‘4.4.0’

In [3]:
# Clean the environment and set the directory
# Don't change anything here
rm(list=ls())
# NOTE: lab cluster path below -- change it to your own path
setwd("/resnick/groups/NolanLab/shared_resources/annotated_objects/")

In [4]:
# Load libraries
suppressMessages(library(Seurat))
suppressMessages(library(scales))
suppressMessages(library(data.table))
suppressMessages(library(tidyverse))
suppressMessages(library(ggplot2))
suppressMessages(library(grid))
suppressMessages(library(SeuratDisk))

In [5]:
# Check the loaded packages and their versions. 
# # Be sure to execute this! It's important for reproducibility and debugging
# Don't change anything here
sessionInfo()

R version 4.4.3 (2025-02-28)
Platform: x86_64-conda-linux-gnu
Running under: Red Hat Enterprise Linux 9.5 (Plow)

Matrix products: default
BLAS/LAPACK: /resnick/groups/NolanLab/cchen/miniconda3/envs/seu4_label_transfer/lib/libopenblasp-r0.3.29.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: America/Los_Angeles
tzcode source: system (glibc)

attached base packages:
[1] grid      stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] SeuratDisk_0.0.0.9021 lubridate_1.9.4       forcats_1.0.0        
 [4] stringr_1.5.1         dplyr_1.1.4           purrr_1.0.4          
 [7] readr_2.1.5       

## Convert Raw counts of lab samples

### Salinity time course data (Newly integrated Seurat 3 samples)

In [6]:
# NOTE: lab cluster path below -- change it to your own path
seu <- read_rds("/resnick/groups/NolanLab/shared_resources/annotated_objects/rds/integrated/Feronia_salinity_timecourse_4S_20251226_seu3_sc291suspiciousClusterRemoved.rds")
print(length(table(seu$time.celltype.anno.Li)))
print(length(table(seu$time.celltype.anno.Li.crude)))


[1] 46
[1] 37


In [7]:
sort(unique(seu$orig.ident))

[1] "sc_291" "sc_292" "sc_293" "sc_294"

In [8]:
sample = "Feronia_salinity_timecourse_4S_20251226_seu3_sc291suspiciousClusterRemoved"

In [9]:
meta.data <-  seu@meta.data
reductions <- seu@reductions$umap@cell.embeddings
write.csv(meta.data,paste0("./h5ad/",sample,"_metadata.csv"))
write.csv(reductions, paste0("./h5ad/",sample,"_umap.csv"))
DefaultAssay(seu) <- "RNA"
seu <- DietSeurat(seu, assays = "RNA")


In [10]:
# Need to replace data with raw counts. SaveH5Seurat only read the data in RNA@data
seu@assays$RNA@data <- seu@assays$RNA@counts
SaveH5Seurat(seu, filename = paste0("./h5ad/", sample,"_RNA.h5Seurat"), overwrite=TRUE)
Convert(paste0("./h5ad/", sample,"_RNA.h5Seurat"), dest = "h5ad", overwrite=TRUE)
system(paste0("rm ./h5ad/",sample,"_RNA.h5Seurat"))

Creating h5Seurat file for version 3.1.5.9900

Adding counts for RNA

Adding data for RNA

No variable features found for RNA

No feature-level metadata found for RNA

Validating h5Seurat file

Adding data from RNA as X

Adding counts from RNA as raw

Transfering meta.data to obs

